# Inscopix Experiments to NWB Format

In [ ]:
import linecache
from pathlib import Path

from tqdm.notebook import tqdm
from neuroconv.datainterfaces import InscopixImagingInterface
import isx

## 1. Path Transversal

In [ ]:
data_dir = Path("/mnt/r2d2/2_Inscopix/1_DTT/")
prefixes = ["1", "2", "3", "4"]
genotypes = ["VGAT", "VGLUT", "OXTR"]
experiments = ["1_OdorAnalysis",  "2_HFvFM", "3_EPM", "4_Social"]
odor_experiment_subtypes = ["1_Concentration", "2_Identity", "4_Social_Odor"]
experiment_dirs = [item for item in data_dir.iterdir() if item.is_dir() and item.name[0] in prefixes]

In [ ]:
def get_animals_per_genotypes(data_dir: Path, subtype_label: str) -> list:
    experiment_animals = []

    if "Social" in subtype_label:
        _genotypes = ["OXTR"]
    else:
        _genotypes = genotypes

    for genotype in _genotypes:
        genotype_dir = data_dir / genotype

        for animal in genotype_dir.glob(f"{genotype}*/"):
            if not animal.is_dir() or "z" in animal.name.lower():
                continue

            animal_dict = {
                "name": animal.name,
                "genotype": genotype,
                "subtype": subtype_label,
                "path": animal
            }

            experiment_animals.append(animal_dict)

    return experiment_animals


In [ ]:
path_dict = {}

for experiment in experiments:
    experiment_dir = data_dir / experiment
    experiment_animals = []
    if experiment == "1_OdorAnalysis":
        # Odor is special; it has subtypes
        for subtype in odor_experiment_subtypes:
            # Cycle Each Subtype
            subtype_dir = experiment_dir / subtype
            subtype_label = subtype.split("_")[1]
            experiment_animals.extend(get_animals_per_genotypes(subtype_dir, subtype_label))

        path_dict[experiment] = experiment_animals

    else:
        subtype_label = experiment.split("_")[1]
        path_dict[experiment] = get_animals_per_genotypes(experiment_dir, subtype_label)